# 00 - Preparazione, organizzazione e validazione dei dataset

Questo notebook implementa la Fase 1 della pipeline sperimentale del progetto SOASEC - The Legend Challenge.

Obiettivi:

1. creare/verificare la struttura delle directory di progetto;
2. caricare i quattro dataset master in formato JSONL;
3. validare struttura, campi obbligatori, duplicati e numerosità;
4. convertire i dataset master in formato chat/instruction compatibile con Hugging Face;
5. generare un manifest dei dataset e un report di validazione.

Questo notebook non addestra ancora il modello. Serve a preparare gli artefatti che verranno usati dal notebook successivo per il training LoRA/QLoRA.


In [110]:
#Setup directory
from pathlib import Path
import json
import csv
from datetime import datetime
from collections import Counter

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

DIRS = {
    "data": PROJECT_ROOT / "data",
    "train": PROJECT_ROOT / "data" / "train",
    "processed": PROJECT_ROOT / "data" / "processed",
    "hf_train": PROJECT_ROOT / "data" / "processed" / "hf_train",
    "results": PROJECT_ROOT / "results",
    "manifests": PROJECT_ROOT / "manifests",
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "regulation": {
        "path": DIRS["train"] / "regulation.jsonl",
        "expected_rows": 50,
        "description": "Dataset di training basato su principi normativi astratti.",
    },
    "positive_legends": {
        "path": DIRS["train"] / "positive_legends.jsonl",
        "expected_rows": 50,
        "description": "Dataset di training basato su positive legends, cioè scenari conformi.",
    },
    "negative_legends": {
        "path": DIRS["train"] / "negative_legends.jsonl",
        "expected_rows": 50,
        "description": "Dataset di training basato su negative legends, cioè scenari non conformi o anti-pattern.",
    },
    "negative_to_correction": {
        "path": DIRS["train"] / "negative_to_correction.jsonl",
        "expected_rows": 50,
        "description": "Dataset di training basato sulla trasformazione da scenario non conforme a correzione conforme.",
    },
}

print("Directory verificate/create correttamente.")

for dataset_name, cfg in DATASETS.items():
    print(f"{dataset_name:24s} -> {cfg['path']}")

PROJECT_ROOT = c:\Users\ab161\Desktop\SoasecProgetto\Project
Directory verificate/create correttamente.
regulation               -> c:\Users\ab161\Desktop\SoasecProgetto\Project\data\train\regulation.jsonl
positive_legends         -> c:\Users\ab161\Desktop\SoasecProgetto\Project\data\train\positive_legends.jsonl
negative_legends         -> c:\Users\ab161\Desktop\SoasecProgetto\Project\data\train\negative_legends.jsonl
negative_to_correction   -> c:\Users\ab161\Desktop\SoasecProgetto\Project\data\train\negative_to_correction.jsonl


In [111]:
# Cella 4 — Funzioni di lettura, validazione e scrittura JSONL/CSV

REQUIRED_FIELDS = {"id", "input", "output"}


def read_jsonl(path: Path):
    """
    Legge un file JSONL.
    Restituisce:
    - records: lista di oggetti JSON validi;
    - errors: lista di errori di parsing.
    """
    records = []
    errors = []

    if not path.exists():
        return records, [{"line": None, "error": f"File non trovato: {path}"}]

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            raw = line.strip()

            if not raw:
                continue

            try:
                obj = json.loads(raw)

                if not isinstance(obj, dict):
                    errors.append({
                        "line": line_number,
                        "error": "La riga non contiene un oggetto JSON."
                    })
                    continue

                obj["_line_number"] = line_number
                records.append(obj)

            except json.JSONDecodeError as exc:
                errors.append({
                    "line": line_number,
                    "error": str(exc),
                })

    return records, errors


def is_valid_master_record(record: dict):
    """
    Un record master è valido se contiene id, input e output,
    e se tutti e tre i campi sono valorizzati.
    """
    if not REQUIRED_FIELDS.issubset(record.keys()):
        return False

    for field in REQUIRED_FIELDS:
        value = record.get(field)

        if value is None or str(value).strip() == "":
            return False

    return True


def validate_records(dataset_name: str, records: list, parsing_errors: list, expected_rows=None):
    """
    Valida i record master di un dataset.

    Controlla:
    - errori JSON;
    - campi obbligatori mancanti;
    - campi obbligatori vuoti;
    - ID duplicati;
    - input duplicati;
    - numerosità attesa;
    - statistiche base su input/output.

    Restituisce:
    - summary;
    - issues;
    - valid_records.
    """
    issues = []

    for err in parsing_errors:
        issues.append({
            "dataset": dataset_name,
            "line": err.get("line"),
            "severity": "ERROR",
            "issue": "json_parse_error",
            "detail": err.get("error"),
        })

    ids = []
    inputs = []
    valid_records = []

    for record in records:
        line = record.get("_line_number")
        row_has_error = False

        missing = REQUIRED_FIELDS - set(record.keys())

        if missing:
            issues.append({
                "dataset": dataset_name,
                "line": line,
                "severity": "ERROR",
                "issue": "missing_required_fields",
                "detail": ", ".join(sorted(missing)),
            })
            row_has_error = True

        for field in REQUIRED_FIELDS:
            value = record.get(field)

            if value is None or str(value).strip() == "":
                issues.append({
                    "dataset": dataset_name,
                    "line": line,
                    "severity": "ERROR",
                    "issue": "empty_required_field",
                    "detail": field,
                })
                row_has_error = True

        if "id" in record and str(record.get("id", "")).strip():
            ids.append(str(record["id"]).strip())

        if "input" in record and str(record.get("input", "")).strip():
            inputs.append(str(record["input"]).strip())

        if not row_has_error:
            valid_records.append(record)

    id_counts = Counter(ids)

    for duplicated_id, count in id_counts.items():
        if duplicated_id and count > 1:
            issues.append({
                "dataset": dataset_name,
                "line": None,
                "severity": "ERROR",
                "issue": "duplicated_id",
                "detail": f"{duplicated_id} appears {count} times",
            })

    input_counts = Counter(inputs)

    for duplicated_input, count in input_counts.items():
        if duplicated_input and count > 1:
            issues.append({
                "dataset": dataset_name,
                "line": None,
                "severity": "WARNING",
                "issue": "duplicated_input",
                "detail": duplicated_input[:180],
            })

    if expected_rows is not None and len(records) != expected_rows:
        issues.append({
            "dataset": dataset_name,
            "line": None,
            "severity": "WARNING",
            "issue": "unexpected_row_count",
            "detail": f"Expected {expected_rows}, found {len(records)}",
        })

    input_lengths = [
        len(str(r.get("input", "")))
        for r in valid_records
    ]

    output_lengths = [
        len(str(r.get("output", "")))
        for r in valid_records
    ]

    summary = {
        "dataset": dataset_name,
        "rows": len(records),
        "valid_rows": len(valid_records),
        "expected_rows": expected_rows,
        "errors": sum(1 for x in issues if x["severity"] == "ERROR"),
        "warnings": sum(1 for x in issues if x["severity"] == "WARNING"),
        "min_input_chars": min(input_lengths) if input_lengths else 0,
        "max_input_chars": max(input_lengths) if input_lengths else 0,
        "avg_input_chars": round(sum(input_lengths) / len(input_lengths), 2) if input_lengths else 0,
        "min_output_chars": min(output_lengths) if output_lengths else 0,
        "max_output_chars": max(output_lengths) if output_lengths else 0,
        "avg_output_chars": round(sum(output_lengths) / len(output_lengths), 2) if output_lengths else 0,
    }

    return summary, issues, valid_records


def write_jsonl(path: Path, records: list):
    """
    Scrive una lista di record in formato JSONL.
    Sovrascrive il file se esiste già.
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def write_csv(path: Path, rows: list, fieldnames: list):
    """
    Scrive una lista di dict in CSV.
    Funziona anche se rows è vuoto.
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for row in rows:
            writer.writerow(row)

In [112]:
# Cella 4 — Funzioni di lettura, validazione e scrittura JSONL

REQUIRED_FIELDS = {"id", "input", "output"}


def read_jsonl(path: Path):
    """
    Legge un file JSONL.
    Restituisce:
    - records: lista di dict validamente parsati;
    - errors: lista di errori di parsing.
    """
    records = []
    errors = []

    if not path.exists():
        return records, [{"line": None, "error": f"File non trovato: {path}"}]

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            raw = line.strip()

            if not raw:
                continue

            try:
                obj = json.loads(raw)

                if not isinstance(obj, dict):
                    errors.append({
                        "line": line_number,
                        "error": "La riga non contiene un oggetto JSON."
                    })
                    continue

                obj["_line_number"] = line_number
                records.append(obj)

            except json.JSONDecodeError as exc:
                errors.append({
                    "line": line_number,
                    "error": str(exc),
                })

    return records, errors


def is_valid_master_record(record: dict):
    """
    Un record master è valido se contiene id, input e output,
    e se tutti e tre i campi sono valorizzati.
    """
    has_required_fields = REQUIRED_FIELDS.issubset(record.keys())

    if not has_required_fields:
        return False

    required_fields_not_empty = all(
        str(record.get(field, "")).strip()
        for field in REQUIRED_FIELDS
    )

    return required_fields_not_empty


def validate_records(dataset_name: str, records: list, parsing_errors: list, expected_rows=None):
    """
    Valida i record master di un dataset.

    Controlla:
    - errori JSON;
    - campi obbligatori mancanti;
    - campi obbligatori vuoti;
    - ID duplicati;
    - input duplicati;
    - numerosità attesa;
    - statistiche base su input/output.
    """
    issues = []

    for err in parsing_errors:
        issues.append({
            "dataset": dataset_name,
            "line": err.get("line"),
            "severity": "ERROR",
            "issue": "json_parse_error",
            "detail": err.get("error"),
        })

    ids = []
    inputs = []
    valid_records = []

    for record in records:
        line = record.get("_line_number")
        row_has_error = False

        missing = REQUIRED_FIELDS - set(record.keys())

        if missing:
            issues.append({
                "dataset": dataset_name,
                "line": line,
                "severity": "ERROR",
                "issue": "missing_required_fields",
                "detail": ", ".join(sorted(missing)),
            })
            row_has_error = True

        for field in REQUIRED_FIELDS:
            value = record.get(field)

            if value is None or str(value).strip() == "":
                issues.append({
                    "dataset": dataset_name,
                    "line": line,
                    "severity": "ERROR",
                    "issue": "empty_required_field",
                    "detail": field,
                })
                row_has_error = True

        if "id" in record and str(record.get("id", "")).strip():
            ids.append(str(record["id"]).strip())

        if "input" in record and str(record.get("input", "")).strip():
            inputs.append(str(record["input"]).strip())

        if not row_has_error:
            valid_records.append(record)

    id_counts = Counter(ids)

    for duplicated_id, count in id_counts.items():
        if duplicated_id and count > 1:
            issues.append({
                "dataset": dataset_name,
                "line": None,
                "severity": "ERROR",
                "issue": "duplicated_id",
                "detail": f"{duplicated_id} appears {count} times",
            })

    input_counts = Counter(inputs)

    for duplicated_input, count in input_counts.items():
        if duplicated_input and count > 1:
            issues.append({
                "dataset": dataset_name,
                "line": None,
                "severity": "WARNING",
                "issue": "duplicated_input",
                "detail": duplicated_input[:180],
            })

    if expected_rows is not None and len(records) != expected_rows:
        issues.append({
            "dataset": dataset_name,
            "line": None,
            "severity": "WARNING",
            "issue": "unexpected_row_count",
            "detail": f"Expected {expected_rows}, found {len(records)}",
        })

    input_lengths = [
        len(str(r.get("input", "")))
        for r in valid_records
    ]

    output_lengths = [
        len(str(r.get("output", "")))
        for r in valid_records
    ]

    summary = {
        "dataset": dataset_name,
        "rows": len(records),
        "valid_rows": len(valid_records),
        "expected_rows": expected_rows,
        "errors": sum(1 for x in issues if x["severity"] == "ERROR"),
        "warnings": sum(1 for x in issues if x["severity"] == "WARNING"),
        "min_input_chars": min(input_lengths) if input_lengths else 0,
        "max_input_chars": max(input_lengths) if input_lengths else 0,
        "avg_input_chars": round(sum(input_lengths) / len(input_lengths), 2) if input_lengths else 0,
        "min_output_chars": min(output_lengths) if output_lengths else 0,
        "max_output_chars": max(output_lengths) if output_lengths else 0,
        "avg_output_chars": round(sum(output_lengths) / len(output_lengths), 2) if output_lengths else 0,
    }

    return summary, issues, valid_records


def write_jsonl(path: Path, records: list):
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def write_csv(path: Path, rows: list, fieldnames: list):
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for row in rows:
            writer.writerow(row)

In [113]:
# Cella 5 — Validazione dei quattro dataset e salvataggio report CSV

all_records = {}
validation_summaries = []
validation_issues = []

for dataset_name, cfg in DATASETS.items():
    records, parsing_errors = read_jsonl(cfg["path"])

    summary, issues, valid_records = validate_records(
        dataset_name=dataset_name,
        records=records,
        parsing_errors=parsing_errors,
        expected_rows=cfg.get("expected_rows"),
    )

    all_records[dataset_name] = valid_records
    validation_summaries.append(summary)
    validation_issues.extend(issues)

print("SUMMARY VALIDAZIONE")
print("-" * 100)

for row in validation_summaries:
    print(row)

print("\nISSUES")
print("-" * 100)

if validation_issues:
    for issue in validation_issues[:80]:
        print(issue)

    if len(validation_issues) > 80:
        print(f"... altri {len(validation_issues) - 80} issue non mostrati a video")
else:
    print("Nessun issue rilevato.")

summary_path = DIRS["results"] / "00_dataset_validation_summary.csv"
issues_path = DIRS["results"] / "00_dataset_validation_issues.csv"

write_csv(
    summary_path,
    validation_summaries,
    fieldnames=[
        "dataset",
        "rows",
        "valid_rows",
        "expected_rows",
        "errors",
        "warnings",
        "min_input_chars",
        "max_input_chars",
        "avg_input_chars",
        "min_output_chars",
        "max_output_chars",
        "avg_output_chars",
    ],
)

write_csv(
    issues_path,
    validation_issues,
    fieldnames=[
        "dataset",
        "line",
        "severity",
        "issue",
        "detail",
    ],
)

print(f"\nCreato: {summary_path}")
print(f"Creato: {issues_path}")

SUMMARY VALIDAZIONE
----------------------------------------------------------------------------------------------------
{'dataset': 'regulation', 'rows': 50, 'valid_rows': 50, 'expected_rows': 50, 'errors': 0, 'warnings': 0, 'min_input_chars': 60, 'max_input_chars': 137, 'avg_input_chars': 94.64, 'min_output_chars': 167, 'max_output_chars': 456, 'avg_output_chars': 263.94}
{'dataset': 'positive_legends', 'rows': 50, 'valid_rows': 50, 'expected_rows': 50, 'errors': 0, 'warnings': 0, 'min_input_chars': 51, 'max_input_chars': 96, 'avg_input_chars': 73.66, 'min_output_chars': 127, 'max_output_chars': 245, 'avg_output_chars': 175.76}
{'dataset': 'negative_legends', 'rows': 50, 'valid_rows': 50, 'expected_rows': 50, 'errors': 0, 'warnings': 0, 'min_input_chars': 45, 'max_input_chars': 110, 'avg_input_chars': 75.26, 'min_output_chars': 136, 'max_output_chars': 222, 'avg_output_chars': 173.1}
{'dataset': 'negative_to_correction', 'rows': 50, 'valid_rows': 50, 'expected_rows': 50, 'errors': 0,

In [114]:
# Cella 6 — Scelta metodologica: niente system prompt nel training

TRAINING_MESSAGE_FORMAT = "user_assistant_only"
TRAINING_USES_SYSTEM_PROMPT = False

print("Formato training scelto:", TRAINING_MESSAGE_FORMAT)
print("System prompt nel training:", TRAINING_USES_SYSTEM_PROMPT)

# Nota metodologica:
# Nel training degli adapter usiamo solo coppie user -> assistant.
# I system prompt saranno usati solo nella fase di evaluation/test,
# in particolare per indirizzare i task GLUE/SuperGLUE-inspired.

Formato training scelto: user_assistant_only
System prompt nel training: False


In [115]:
# Cella 7 — Conversione in formato Hugging Face chat senza system prompt

def to_hf_chat_record(dataset_name: str, record: dict):
    """
    Converte un record master input/output in formato chat.

    Scelta metodologica:
    - training: solo user -> assistant;
    - niente role=system nel training;
    - i system prompt verranno usati solo nella fase di evaluation/test.
    """
    return {
        "id": str(record["id"]).strip(),
        "dataset": dataset_name,
        "messages": [
            {
                "role": "user",
                "content": str(record["input"]).strip(),
            },
            {
                "role": "assistant",
                "content": str(record["output"]).strip(),
            },
        ],
        "source_line": record.get("_line_number"),
    }


hf_dataset_paths = {}
hf_records_by_dataset = {}

for dataset_name, records in all_records.items():
    hf_records = [
        to_hf_chat_record(dataset_name, record)
        for record in records
        if is_valid_master_record(record)
    ]

    output_path = DIRS["hf_train"] / f"{dataset_name}_hf_chat.jsonl"

    write_jsonl(output_path, hf_records)

    hf_dataset_paths[dataset_name] = output_path
    hf_records_by_dataset[dataset_name] = hf_records

    print(f"{dataset_name:24s}: {len(hf_records)} record -> {output_path}")

regulation              : 50 record -> c:\Users\ab161\Desktop\SoasecProgetto\Project\data\processed\hf_train\regulation_hf_chat.jsonl
positive_legends        : 50 record -> c:\Users\ab161\Desktop\SoasecProgetto\Project\data\processed\hf_train\positive_legends_hf_chat.jsonl
negative_legends        : 50 record -> c:\Users\ab161\Desktop\SoasecProgetto\Project\data\processed\hf_train\negative_legends_hf_chat.jsonl
negative_to_correction  : 50 record -> c:\Users\ab161\Desktop\SoasecProgetto\Project\data\processed\hf_train\negative_to_correction_hf_chat.jsonl


In [116]:
# Cella 8 — Dataset combinato opzionale

combined_records = []

for dataset_name in DATASETS.keys():
    combined_records.extend(
        hf_records_by_dataset.get(dataset_name, [])
    )

combined_path = DIRS["hf_train"] / "all_training_hf_chat.jsonl"

write_jsonl(combined_path, combined_records)

print(f"Creato dataset combinato opzionale: {combined_path}")
print(f"Totale record combinati: {len(combined_records)}")

Creato dataset combinato opzionale: c:\Users\ab161\Desktop\SoasecProgetto\Project\data\processed\hf_train\all_training_hf_chat.jsonl
Totale record combinati: 200


In [117]:
# Cella 9 — Manifest dei dataset

manifest = {
    "project": "SOASEC - The Legend Challenge",
    "phase": "00_prepare_datasets",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "training_message_format": TRAINING_MESSAGE_FORMAT,
    "training_uses_system_prompt": TRAINING_USES_SYSTEM_PROMPT,
    "datasets": {},
    "reports": {
        "validation_summary_csv": str(summary_path.relative_to(PROJECT_ROOT)),
        "validation_issues_csv": str(issues_path.relative_to(PROJECT_ROOT)),
    },
    "combined_optional_dataset": str(combined_path.relative_to(PROJECT_ROOT)),
}

summary_by_name = {
    row["dataset"]: row
    for row in validation_summaries
}

for dataset_name, cfg in DATASETS.items():
    source_path = cfg["path"]
    processed_path = hf_dataset_paths.get(dataset_name)

    manifest["datasets"][dataset_name] = {
        "description": cfg["description"],
        "source_jsonl": str(source_path.relative_to(PROJECT_ROOT)),
        "processed_hf_chat_jsonl": str(processed_path.relative_to(PROJECT_ROOT)) if processed_path else None,
        "expected_rows": cfg.get("expected_rows"),
        "actual_rows": summary_by_name.get(dataset_name, {}).get("rows"),
        "valid_rows": summary_by_name.get(dataset_name, {}).get("valid_rows"),
        "errors": summary_by_name.get(dataset_name, {}).get("errors"),
        "warnings": summary_by_name.get(dataset_name, {}).get("warnings"),
        "message_format": "user_assistant_only",
    }

manifest_path = DIRS["manifests"] / "00_dataset_manifest.json"

with manifest_path.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f"Creato manifest: {manifest_path}")

Creato manifest: c:\Users\ab161\Desktop\SoasecProgetto\Project\manifests\00_dataset_manifest.json


In [118]:
# Cella 10 — Controllo finale bloccante

total_errors = sum(row["errors"] for row in validation_summaries)
total_warnings = sum(row["warnings"] for row in validation_summaries)

expected_total_records = sum(
    cfg.get("expected_rows", 0)
    for cfg in DATASETS.values()
    if cfg.get("expected_rows") is not None
)

actual_total_records = len(combined_records)

system_messages_count = sum(
    1
    for record in combined_records
    for message in record.get("messages", [])
    if message.get("role") == "system"
)

invalid_role_patterns = []

for record in combined_records:
    roles = tuple(message.get("role") for message in record.get("messages", []))

    if roles != ("user", "assistant"):
        invalid_role_patterns.append({
            "id": record.get("id"),
            "dataset": record.get("dataset"),
            "roles": roles,
        })

print(f"Errori totali: {total_errors}")
print(f"Warning totali: {total_warnings}")
print(f"Record HF totali creati: {actual_total_records}")
print(f"Record HF attesi: {expected_total_records}")
print(f"System messages nel training: {system_messages_count}")
print(f"Record con pattern ruoli non valido: {len(invalid_role_patterns)}")

if total_errors > 0:
    print("\nBLOCCANTE: correggere gli errori prima di passare al training.")
elif expected_total_records and actual_total_records != expected_total_records:
    print("\nBLOCCANTE: numero di record HF diverso da quello atteso.")
elif system_messages_count > 0:
    print("\nBLOCCANTE: sono ancora presenti messaggi system nel training.")
elif invalid_role_patterns:
    print("\nBLOCCANTE: alcuni record non hanno il formato user -> assistant.")
    print("Primi record problematici:")
    for row in invalid_role_patterns[:10]:
        print(row)
elif total_warnings > 0:
    print("\nATTENZIONE: dataset validati, ma ci sono warning da valutare.")
else:
    print("\nOK: dataset validati. Puoi passare alla fase di training degli adapter.")

Errori totali: 0
Warning totali: 0
Record HF totali creati: 200
Record HF attesi: 200
System messages nel training: 0
Record con pattern ruoli non valido: 0

OK: dataset validati. Puoi passare alla fase di training degli adapter.


In [119]:
# Cella 11 — Esempio record finale e controllo pattern ruoli

role_patterns = Counter()

for record in combined_records:
    roles = tuple(message.get("role") for message in record.get("messages", []))
    role_patterns[roles] += 1

print("Pattern ruoli nel dataset combinato:")
print("-" * 100)

for pattern, count in role_patterns.items():
    print(f"{pattern}: {count}")

print("\nEsempio record finale:")
print("-" * 100)

if combined_records:
    print(json.dumps(combined_records[0], ensure_ascii=False, indent=2))
else:
    print("Nessun record disponibile. Inserisci i JSONL master in data/train/ e riesegui il notebook.")

Pattern ruoli nel dataset combinato:
----------------------------------------------------------------------------------------------------
('user', 'assistant'): 200

Esempio record finale:
----------------------------------------------------------------------------------------------------
{
  "id": "R1.L1.DR.01",
  "dataset": "regulation",
  "messages": [
    {
      "role": "user",
      "content": "What does the Gender Equality Strategy say about safety in workplaces, public spaces, and online environments?"
    },
    {
      "role": "assistant",
      "content": "The strategy states that everyone should be safe in their homes, close relationships, workplaces, public spaces, and online. It also links this safety to the freedom to express ideas and emotions and to pursue educational and professional paths without the constraints of stereotypical gender norms."
    }
  ],
  "source_line": 1
}
